<a href="https://colab.research.google.com/github/Jacksonfvsantos/lol-worlds-predictor/blob/main/notebooks/worlds_predictor_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import os
import subprocess
import warnings

os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

subprocess.run(
    ['pip', 'install', 'urllib3<2', 'mwclient', 'joblib', 'scikit-learn', 'pandas', 'numpy', '-q'],
    capture_output=True
)

!pip install "urllib3<2" mwclient joblib scikit-learn pandas numpy -q

import glob
import pandas as pd
import numpy as np
import mwclient
import joblib
from google.colab import drive

drive.mount('/content/drive')

if not os.path.exists('/content/lol-worlds-predictor'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Jacksonfvsantos/lol-worlds-predictor.git'],
        capture_output=True
    )
print("✅ Ambiente configurado, Repositório clonado e Google Drive conectado!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Ambiente configurado, Repositório clonado e Google Drive conectado!


In [13]:
caminho_dados_drive = '/content/drive/MyDrive/lol-worlds-predictor-data/data/*.csv'
caminho_local = '/content/lol-worlds-predictor/data/*.csv'
arquivos_csv = glob.glob(caminho_dados_drive) if glob.glob(caminho_dados_drive) else glob.glob(caminho_local)

if not arquivos_csv:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado! Verifique se os arquivos de 2018 a 2026 estão na pasta 'lol-worlds-predictor-data/data/' do seu Drive.")

lista_dfs = [pd.read_csv(arquivo, low_memory=False) for arquivo in arquivos_csv]
df_lol = pd.concat(lista_dfs, ignore_index=True)

print(f"✅ {len(arquivos_csv)} arquivo(s) CSV carregado(s) com sucesso! Total de linhas: {len(df_lol)}")

✅ 9 arquivo(s) CSV carregado(s) com sucesso! Total de linhas: 1084956


In [14]:
df_times = df_lol[df_lol['position'] == 'team'].copy()

df_features = df_times.groupby('teamname').agg(
    total_jogos=('gameid', 'count'),
    winrate_temporada=('result', 'mean'),
    golddiffat15=('golddiffat15', 'mean'),
    firstblood=('firstblood', 'mean'),
    firstdragon=('firstdragon', 'mean'),
    firsttower=('firsttower', 'mean'),
    vspm=('vspm', 'mean')
).reset_index()

df_features = df_features[df_features['total_jogos'] >= 15].reset_index(drop=True)

print(f"✅ Perfil gerado para {len(df_features)} equipes!")
display(df_features.head(5))

✅ Perfil gerado para 1380 equipes!


,teamname,total_jogos,winrate_temporada,golddiffat15,firstblood,firstdragon,firsttower,vspm
0,100 Thieves,508,0.513780,-166.761811,0.490157,0.507874,0.500000,7.871707
1,100 Thieves Academy,356,0.567416,214.076056,0.466292,0.498592,0.500000,7.030099
2,100 Thieves Challengers,40,0.600000,297.200000,0.575000,0.425000,0.550000,7.486728
3,100 Thieves Next,187,0.673797,1378.229947,0.604278,0.449198,0.561497,6.630823
4,19esports,27,0.629630,1802.740741,0.629630,0.370370,0.814815,7.711070


In [15]:
df_blue = df_times[df_times['side'] == 'Blue'][['gameid', 'teamname', 'result']]
df_red = df_times[df_times['side'] == 'Red'][['gameid', 'teamname']]

df_confrontos = pd.merge(df_blue, df_red, on='gameid', suffixes=('_blue', '_red'))

df_features_blue = df_features.rename(columns=lambda x: x + '_blue' if x != 'teamname' else 'teamname_blue')
df_features_red = df_features.rename(columns=lambda x: x + '_red' if x != 'teamname' else 'teamname_red')

df_modelo = pd.merge(df_confrontos, df_features_blue, on='teamname_blue')
df_modelo = pd.merge(df_modelo, df_features_red, on='teamname_red')

df_modelo['diff_winrate'] = df_modelo['winrate_temporada_blue'] - df_modelo['winrate_temporada_red']
df_modelo['diff_golddiffat15'] = df_modelo['golddiffat15_blue'] - df_modelo['golddiffat15_red']
df_modelo['diff_firstblood'] = df_modelo['firstblood_blue'] - df_modelo['firstblood_red']
df_modelo['diff_firstdragon'] = df_modelo['firstdragon_blue'] - df_modelo['firstdragon_red']
df_modelo['diff_firsttower'] = df_modelo['firsttower_blue'] - df_modelo['firsttower_red']
df_modelo['diff_vspm'] = df_modelo['vspm_blue'] - df_modelo['vspm_red']

colunas_finais = [
    'gameid', 'teamname_blue', 'teamname_red', 'result',
    'diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
    'diff_firstdragon', 'diff_firsttower', 'diff_vspm'
]

df_treino = df_modelo[colunas_finais].copy()
df_treino.dropna(inplace=True)

print(f"✅ Dataset de treino criado com {len(df_treino)} confrontos!")

✅ Dataset de treino criado com 84180 confrontos!


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

features = [
    'diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
    'diff_firstdragon', 'diff_firsttower', 'diff_vspm'
]
X = df_treino[features]
y = df_treino['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_lol = LogisticRegression(max_iter=1000)
modelo_lol.fit(X_train, y_train)

y_pred = modelo_lol.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)

print(f"🏆 Acurácia Geral do Modelo: {acuracia:.2%}\n")

pasta_modelos = '/content/drive/MyDrive/lol-worlds-predictor-data/models'
os.makedirs(pasta_modelos, exist_ok=True)

caminho_modelo = f'{pasta_modelos}/modelo_lol_worlds.pkl'
joblib.dump(modelo_lol, caminho_modelo)
print(f"✅ Modelo salvo em segurança: {caminho_modelo}")

🏆 Acurácia Geral do Modelo: 62.98%

✅ Modelo salvo em segurança: /content/drive/MyDrive/lol-worlds-predictor-data/models/modelo_lol_worlds.pkl


In [17]:
alias_times = {
    "Anyone's Legend": "AL",
    "Bilibili Gaming": "BLG",
    "CTBC Flying Oyster": "CFO",
    "Dplus KIA": "DK",
    "G2 Esports": "G2",
    "Gen.G":"GEN",
    "Hanwha Life Esports": "HLE",
    "Invictus Gaming": "iG",
    "Movistar KOI": "MKOI",
    "Team Liquid": "TL",
    "Team Secret Whales": "TSW",
    "Top Esports": "TES",
    "Cloud9": "C9",
    "Karmine Corp": "KC",
    "MVK Esports": "MVK",
    "FURIA": "FUR"
}

def prever_confronto(time_azul, time_vermelho, df_stats, modelo):
    time_azul_corr = alias_times.get(time_azul, time_azul)
    time_vermelho_corr = alias_times.get(time_vermelho, time_vermelho)

    stats_azul = df_stats[df_stats['teamname'] == time_azul_corr]
    stats_vermelho = df_stats[df_stats['teamname'] == time_vermelho_corr]

    if stats_azul.empty or stats_vermelho.empty:
        print(f"❌ Erro: '{time_azul_corr}' ou '{time_vermelho_corr}' sem dados no histórico.")
        return None

    diff_winrate = float(stats_azul['winrate_temporada'].values[0] - stats_vermelho['winrate_temporada'].values[0])
    diff_golddiffat15 = float(stats_azul['golddiffat15'].values[0] - stats_vermelho['golddiffat15'].values[0])
    diff_firstblood = float(stats_azul['firstblood'].values[0] - stats_vermelho['firstblood'].values[0])
    diff_firstdragon = float(stats_azul['firstdragon'].values[0] - stats_vermelho['firstdragon'].values[0])
    diff_firsttower = float(stats_azul['firsttower'].values[0] - stats_vermelho['firsttower'].values[0])
    diff_vspm = float(stats_azul['vspm'].values[0] - stats_vermelho['vspm'].values[0])

    X_novo = pd.DataFrame([[
        diff_winrate, diff_golddiffat15, diff_firstblood,
        diff_firstdragon, diff_firsttower, diff_vspm
    ]], columns=[
        'diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
        'diff_firstdragon', 'diff_firsttower', 'diff_vspm'
    ])

    probabilidades = modelo.predict_proba(X_novo)[0]
    prob_vermelho = probabilidades[0] * 100
    prob_azul = probabilidades[1] * 100

    vencedor = time_azul_corr if prob_azul > prob_vermelho else time_vermelho_corr

    return {
        'Blue Side': time_azul_corr,
        'Red Side': time_vermelho_corr,
        'Chance Blue (%)': round(prob_azul, 2),
        'Chance Red (%)': round(prob_vermelho, 2),
        'Vencedor Previsto': vencedor
    }

print("✅ Função de previsão pronta!")

✅ Função de previsão pronta!


In [18]:
import time
import mwclient
import pandas as pd

site = mwclient.Site(
    'lol.fandom.com',
    path='/',
    clients_useragent='lol-worlds-predictor-tcc/1.0 (Projeto de TCC; Unijorge)'
)

torneio_alvo = "2026 Season World Championship"

max_tentativas = 3
resposta_partidas = None

for tentativa in range(max_tentativas):
    try:
        resposta_partidas = site.api('cargoquery',
            limit='max',
            tables="MatchSchedule",
            fields="Team1, Team2, Tab",
            where=f"OverviewPage='{torneio_alvo}' AND Team1 IS NOT NULL AND Team2 IS NOT NULL",
            format="json"
        )
        print("✅ Dados importados da Leaguepedia com sucesso!")
        break

    except Exception as e:
        print(f"⚠️ Aviso de limite de rede (Tentativa {tentativa + 1}/{max_tentativas}).")
        if tentativa < max_tentativas - 1:
            print("⏳ Pausando por 15 segundos para esfriar o IP...")
            time.sleep(15)
        else:
            print("❌ Falha crítica: Servidor da Fandom indisponível no momento.")

if resposta_partidas:
    lista_partidas = []
    for item in resposta_partidas['cargoquery']:
        dados = item['title']
        lista_partidas.append({
            'Fase_do_Torneio': dados['Tab'],
            'Time_Azul': dados['Team1'],
            'Time_Vermelho': dados['Team2']
        })

    df_chaveamento = pd.DataFrame(lista_partidas)

    print(f"✅ {len(df_chaveamento)} partidas processadas e prontas para simulação!")
    display(df_chaveamento.head())

⚠️ Aviso de limite de rede (Tentativa 1/3).
⏳ Pausando por 15 segundos para esfriar o IP...
⚠️ Aviso de limite de rede (Tentativa 2/3).
⏳ Pausando por 15 segundos para esfriar o IP...
⚠️ Aviso de limite de rede (Tentativa 3/3).
❌ Falha crítica: Servidor da Fandom indisponível no momento.


In [10]:
resultados_simulacao = []

for index, row in df_chaveamento.iterrows():
    res = prever_confronto(row['Time_Azul'], row['Time_Vermelho'], df_features, modelo_lol)
    if res:
        res['Fase'] = row['Fase_do_Torneio']
        resultados_simulacao.append(res)

df_previsoes_finais = pd.DataFrame(resultados_simulacao)

print("🚀 SIMULAÇÃO COMPLETA CONCLUÍDA!")
display(df_previsoes_finais)

🚀 SIMULAÇÃO COMPLETA CONCLUÍDA!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

""


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag